Import packages

In [2]:
import pandas as pd
from ydata_profiling import ProfileReport
import datamart_profiler
import openai
from dotenv import load_dotenv
from openai import OpenAI
import os
import json

Import the dataset and load it into a dataframe

In [3]:
#datasets/kingburrito666/cannabis-strains/versions/9/cannabis.csv già caricato nel db

file_path = 'datasets/akshaydattatraykhare/diabetes-dataset/versions/1/diabetes.csv'

df = pd.read_csv(file_path)

Extract Data Profile using Datamart_profiler, a library developed for Auctus. Auctus is a dataset search engine and data augmentation platform developed at New York University.

It is optimised for Auctus, but used as well for dataset search. We can use it as a base.

In [4]:
metadata = datamart_profiler.process_dataset(df)

C:\Users\ricca\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\datamart_profiler\core.py:199: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = data.astype(object).fillna('').astype(str)


In [5]:
metadata

{'nb_rows': 768,
 'nb_profiled_rows': 768,
 'nb_columns': 9,
 'columns': [{'name': 'Pregnancies',
   'structural_type': 'http://schema.org/Integer',
   'semantic_types': [],
   'unclean_values_ratio': 0.0,
   'num_distinct_values': 17,
   'mean': 3.8450520833333335,
   'stddev': 3.3673836124089886,
   'coverage': [{'range': {'gte': 0.0, 'lte': 3.0}},
    {'range': {'gte': 4.0, 'lte': 6.0}},
    {'range': {'gte': 7.0, 'lte': 13.0}}]},
  {'name': 'Glucose',
   'structural_type': 'http://schema.org/Integer',
   'semantic_types': [],
   'unclean_values_ratio': 0.0,
   'num_distinct_values': 136,
   'mean': 120.89453125,
   'stddev': 31.95179590820272,
   'coverage': [{'range': {'gte': 68.0, 'lte': 108.0}},
    {'range': {'gte': 111.0, 'lte': 146.0}},
    {'range': {'gte': 151.0, 'lte': 196.0}}]},
  {'name': 'BloodPressure',
   'structural_type': 'http://schema.org/Integer',
   'semantic_types': [],
   'unclean_values_ratio': 0.0,
   'num_distinct_values': 47,
   'mean': 69.10546875,
   'st

In [6]:
profile_summary = []

# Iterate through each column in the metadata and summarize the details
for column_meta in metadata['columns']:
    column_summary = f"**{column_meta['name']}**: "
    
    # Structural type
    structural_type = column_meta.get('structural_type', 'Unknown')
    column_summary += f"Data is of type {structural_type.split('/')[-1].lower()}. "

    # Number of distinct values (if applicable)
    if 'num_distinct_values' in column_meta:
        num_distinct_values = column_meta['num_distinct_values']
        column_summary += f"There are {num_distinct_values} unique values. "

    # Check if the column is numeric
    if pd.api.types.is_numeric_dtype(df[column_meta['name']]):
        column_summary += "This column is numeric. " 
        mean_value = df[column_meta['name']].mean()
        max_value = df[column_meta['name']].max()
        min_value = df[column_meta['name']].min()
        column_summary += f"Mean: {mean_value}, Max: {max_value}, Min: {min_value}. "
    elif pd.api.types.is_datetime64_any_dtype(df[column_meta['name']]):
        column_summary += "This column is of datetime type. "
        # Calculate and add the range of dates
        min_date = df[column_meta['name']].min()
        max_date = df[column_meta['name']].max()
        column_summary += f"Date range: from {min_date} to {max_date}. "
    else:
        # For categorical columns, get the top 3 most frequent categories
        value_counts = df[column_meta['name']].value_counts()
        if value_counts.nunique() > 1:
            top_categories = df[column_meta['name']].value_counts().nlargest(3).index.tolist()
            column_summary += f"Top 3 frequent values: {', '.join(top_categories)}. "

    # Handle coverage (if available)
    if 'coverage' in column_meta:
        low=0
        high=0
        for i in range(len(column_meta['coverage'])):
            if(column_meta['coverage'][i]['range']['gte']<low):
                low=column_meta['coverage'][i]['range']['gte']
            if(column_meta['coverage'][i]['range']['lte']>high):
                high=column_meta['coverage'][i]['range']['lte']
        column_summary += f"Coverage spans from {low} to {high}. "

    # Append the summarized profile for this column
    profile_summary.append(column_summary)

dataset_title = os.path.splitext(os.path.basename(file_path))[0]

final_profile_summary = "The key data profile information for the dataset "+ dataset_title +" includes:\n" + '\n'.join(profile_summary)

print(final_profile_summary)

The key data profile information for the dataset diabetes includes:
**Pregnancies**: Data is of type integer. There are 17 unique values. This column is numeric. Mean: 3.8450520833333335, Max: 17, Min: 0. Coverage spans from 0 to 13.0. 
**Glucose**: Data is of type integer. There are 136 unique values. This column is numeric. Mean: 120.89453125, Max: 199, Min: 0. Coverage spans from 0 to 196.0. 
**BloodPressure**: Data is of type integer. There are 47 unique values. This column is numeric. Mean: 69.10546875, Max: 122, Min: 0. Coverage spans from 0 to 100.0. 
**SkinThickness**: Data is of type integer. There are 51 unique values. This column is numeric. Mean: 20.536458333333332, Max: 99, Min: 0. Coverage spans from 0 to 50.0. 
**Insulin**: Data is of type integer. There are 186 unique values. This column is numeric. Mean: 79.79947916666667, Max: 846, Min: 0. Coverage spans from 0 to 271.0. 
**BMI**: Data is of type float. This column is numeric. Mean: 31.992578124999998, Max: 67.1, Min:

Now we analyze the schema and the semantic value of every column to generate a semantic profile

In [7]:
TEMPLATE = """
    {'Temporal': 
        {
            'isTemporal': Does this column contain temporal information? Yes or No,
            'resolution': If Yes, specify the resolution (Year, Month, Day, Hour, etc.).
        },
     'Spatial': {'isSpatial': Does this column contain spatial information? Yes or No,
                 'resolution': If Yes, specify the resolution (Country, State, City, Coordinates, etc.).},
     'Entity Type': What kind of entity does the column describe? (e.g., Person, Location, Organization, Product),
     'Domain-Specific Types': What domain is this column from (e.g., Financial, Healthcare, E-commerce, Climate, Demographic),
     'Function/Usage Context': How might the data be used (e.g., Aggregation Key, Ranking/Scoring, Interaction Data, Measurement).}
    """

RESPONSE_EXAMPLE = """
    {
    "Domain-Specific Types": "General",
    "Entity Type": "Temporal Entity",
    "Function/Usage Context": "Aggregation Key",
    "Spatial": {"isSpatial": false,
                "resolution": ""},
    "Temporal": {"isTemporal": true,
                "resolution": "Year"}
    }
    """

In [8]:
#qui lavoriamo la singola colonna!

def generate_prompt(column_name, sample_values, TEMPLATE, RESPONSE_EXAMPLE):
    prompt = f"""
        You are a dataset semantic analyzer. Based on the column name and sample values, classify the column into multiple semantic types. 
        Please group the semantic types under the following categories: 
        'Temporal', 'Spatial', 'Entity Type', 'Data Format', 'Domain-Specific Types', 'Function/Usage Context'. 
        Following is the template {TEMPLATE}
        Please follow these rules:
        1. The output must be a valid JSON object that can be directly loaded by json.loads. Example response is {RESPONSE_EXAMPLE}
        2. All keys from the template must be present in the response.
        3. All keys and string values must be enclosed in double quotes.
        4. There must be no trailing commas.
        5. Use booleans (true/false) and numbers without quotes.
        6. Do not include any additional information or context in the response.
        7. If you are unsure about a specific category, you can leave it as an empty string.

        Column name: {column_name}
        Sample values: {sample_values}
        """
    return prompt

In [9]:
#setup OpenAI API
load_dotenv()
client = OpenAI()

Prompt = []

In [10]:
#OpenAI call
def call_openai_api(prompt, model):
    if model == 'o1-mini':
        messages = [
            {
                "role": "user",
                "content": prompt
            }
        ]
    else:
        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {
                "role": "user",
                "content": prompt
            }
        ]

    completion = client.chat.completions.create(
    model=model,
    messages=messages
)
    return completion

In [11]:
# Extraction of the column names
column_names = [column_meta['name'] for column_meta in metadata['columns']]

# Samples for a column
def generate_samples(column_name):
    sample_values = df[column_name].dropna().sample(5).tolist()
    return sample_values

In [ ]:
semantic_profile = []

for column in column_names:
    sample_values = generate_samples(column)
    prompt = generate_prompt(column, sample_values, TEMPLATE, RESPONSE_EXAMPLE)
    response = call_openai_api(prompt, "o3-mini")
    semantic_profile.append(response.choices[0].message.content)

#print(semantic_profile)


In [13]:
semantic_profile

['```json\n{\n    "Temporal": {\n        "isTemporal": false,\n        "resolution": ""\n    },\n    "Spatial": {\n        "isSpatial": false,\n        "resolution": ""\n    },\n    "Entity Type": "",\n    "Domain-Specific Types": "Healthcare",\n    "Function/Usage Context": "Measurement"\n}\n```',
 '```json\n{\n    "Temporal": {\n        "isTemporal": false,\n        "resolution": ""\n    },\n    "Spatial": {\n        "isSpatial": false,\n        "resolution": ""\n    },\n    "Entity Type": "",\n    "Domain-Specific Types": "Healthcare",\n    "Function/Usage Context": "Measurement",\n    "Entity Type": "Biomarker"\n}\n```',
 '```json\n{\n    "Temporal": {\n        "isTemporal": false,\n        "resolution": ""\n    },\n    "Spatial": {\n        "isSpatial": false,\n        "resolution": ""\n    },\n    "Entity Type": "Measurement",\n    "Domain-Specific Types": "Healthcare",\n    "Function/Usage Context": "Measurement"\n}\n```',
 '```json\n{\n    "Temporal": {\n        "isTemporal": f

In [14]:
RESPONSE_QUERIES = """ 
{
  "queries": [
    {
      "query": "Select data ..."
    },
    {
      "query": "Find datasets ..."
    },
    {
      "query": "Show me data ..."
    }
  ]
}
"""

In [15]:
prompt_to_instructions = f"""
You are the dataset owner of {dataset_title}. You need to provide instructions to the users on how to discover effectively the dataset in a DataSpace platform.
The user is provided with a prompt interface, so it can ask naturakl language questions to find the dataset.
The dataset contains the following semantic types: {semantic_profile} /n
The data profile is the following: {final_profile_summary} /n   
The final users don't have access to the dataset content. So, provide instructions (queries) that they could use to find this dataset.

One example of a query could be: "Find a dataset with entries about cannabis strains and their effects"

Generate as many queries as required to cover the entire dataset content and structure.
Reason step by step:
1. First understand the dataset content and structure.
2. Formulate general queries to show the dataset content.
3. Formulate precise queries to highlight specific findings or limitations.
4. Formulate queries that consider interactions between different columns.
5. Formulate queries that consider the dataset's temporal and spatial aspects.

The output must be a valid JSON object that can be directly loaded by json.loads. It should be a list of queries. An example response is: {RESPONSE_QUERIES}
"""


In [16]:
list_Q = call_openai_api(prompt_to_instructions, "o1-mini").choices[0].message.content

In [17]:
list_Q = list_Q.replace("```json\n", "").replace("\n```", "")

queries_dict = json.loads(list_Q)

In [18]:
for i, query in enumerate(queries_dict['queries'], start=1):
    print(f"{i}. {query['query']}\n")

1. Find a healthcare dataset that includes measurements related to diabetes.

2. Search for a dataset containing information on pregnancies, glucose levels, and diabetes outcomes.

3. Show me a dataset with numeric health metrics such as BMI, insulin, and blood pressure for diabetes analysis.

4. Locate a dataset that includes the Diabetes Pedigree Function and age of participants.

5. Find a dataset related to diabetes that does not include any temporal or spatial data.

6. Search for a healthcare dataset with columns for skin thickness, glucose, and insulin measurements.

7. Show datasets that include numerical values for age ranging from 21 to 81 years related to diabetes.

8. Find a dataset that tracks the relationship between BMI and diabetes outcome.

9. Look for a healthcare dataset focused on diabetes with variables like blood pressure and pregnancies.

10. Search for a dataset containing measurements for diabetes without any location-based information.

11. Find a diabetes dat

Now let's put the questions in a vector db and try to query to see what happens: 

In [1]:
import chromadb

In [6]:
def create_embeddings_openai(texts):
    response = openai.embeddings.create(
        model="text-embedding-3-small",  
        input=texts
    )
    
    embeddings = [item.embedding for item in response.data]
    return embeddings

class OpenAIEmbeddingFunction:
    def __call__(self, input):
        embeddings = create_embeddings_openai(input)
        return embeddings
    


In [186]:
# Check if an instance of Chroma already exists and reset if necessary
try:
    chroma_client = chromadb.PersistentClient()
    chroma_client.get_settings().allow_reset = True
    #chroma_client.reset()
except ValueError as e:
    print(f"Resetting existing Chroma instance: {e}")
    chroma_client.reset()

embedding_function = OpenAIEmbeddingFunction()

collection = chroma_client.get_or_create_collection(
    name="questions_collection",
    embedding_function=embedding_function
)

Resetting existing Chroma instance: An instance of Chroma already exists for ./chroma with different settings


In [215]:
# Aggiungere documenti alla collection
for i, query in enumerate(queries_dict['queries']):
    collection.add(
        documents=query['query'],
        ids=[f"query_{i}_{dataset_title}"],
        metadatas={"dataset": dataset_title}
    )

print(f"Added {len(queries_dict['queries'])} questions to the vector database.")

Added 15 questions to the vector database.


In [197]:
def query_to_vector_db(query_text, n=3):
    results = collection.query(
        query_texts=[query_text],
        n_results=n
    )

    print(f"\nQuery: {query}")
    print(f"Top {n} results:")
    for i, doc in enumerate(results['documents'][0]):
        print(f"{i+1}. {doc} (ID: {results['ids'][0][i]}, Distance: {results['distances'][0][i]:.4f})")

In [237]:
query = "I need data which contains information about the relations about diabetes and age of the patient."

query_to_vector_db(query, 3)


Query: I need data which contains information about the relations about diabetes and age of the patient.
Top 3 results:
1. Find medical record datasets that include age and pregnancy information for diabetes studies. (ID: query_9_diabetes, Distance: 0.6559)
2. Find healthcare datasets related to diabetes measurements. (ID: query_0_diabetes, Distance: 0.6740)
3. Show me datasets with health metrics for diabetic patients. (ID: query_2_diabetes, Distance: 0.7146)


Now we generate a report for the dataset, later we'll use reports from all datasets to generate MKS

In [25]:
prompt_to_report = f"""
You are the dataset curator of {dataset_title}. You need to write a meta knowledge summary about the the topic of the dataset.
This summary will be used to augment the queries of users looking for datasets, they may not be experts about the topic.

Write a summary about the topic of the dataset with your knowledge and enrich it with key insights you get from the dataset.
Explain what's in the dataset and especially why it is in the dataset.
For example, for a dataset about parkinson you will write a summary about parkinson. While doing it will add to your knowledge the insights you get from the dataset,
for example that a particular medicine is used to treat parkinson. 

The dataset contains the following semantic types: {semantic_profile} /n
The data profile is the following: {final_profile_summary} /n   

The output should be a paragraph, avoid bullet points.

"""

In [26]:
report = call_openai_api(prompt_to_report, "o1-mini").choices[0].message.content

In [27]:
print(report)

Diabetes is a chronic medical condition that affects how the body processes blood sugar (glucose), leading to elevated glucose levels that can result in serious health complications if not managed properly. The diabetes dataset is meticulously curated to encompass a range of health-related measurements that are crucial for understanding and predicting the onset of diabetes. This dataset includes demographic and physiological attributes such as the number of pregnancies, which averages around four, indicating its relevance in assessing risk factors linked to pregnancy-related hormonal changes. Glucose levels, with an average value of approximately 121, are a primary indicator of diabetes, while blood pressure and skin thickness measurements provide additional insights into a patient’s overall health status. Insulin levels in the dataset show significant variability, highlighting the complexity of insulin resistance and secretion in diabetic patients. Body Mass Index (BMI) averages nearl

Now we generate a topic for each dataset. Later on when we receive a query from the user, we detect which are the main n topics for the query and then we retrieve the most interesting/similar reports and we use them to augment the query. A task for later will be to decide how specific the topic has to be.

In [234]:
prompt_to_report = f"""
You are the dataset curator of {dataset_title}. You need to decide which is the topic of this dataset.
The dataset contains the following semantic types: {semantic_profile} /n
The data profile is the following: {final_profile_summary} /n   
The final users need this tag to better find this dataset during a research.

Rules:
1. Don't use too general topics like "science" 
2. Don't use too specific topics like "Lebron James stats"
3. Use a topic that is specific enough to be meaningful but general enough to be relevant to a broad audience.

Example output: "Machine learning in healthcare", "Climate change in the US", "E-commerce trends in 2022", etc.

The output should be only the topic name, no additional text.

"""

In [235]:
topic = call_openai_api(prompt_to_report, "o1-mini").choices[0].message.content

In [236]:
topic

'Diabetes Health Metrics'

To understand how our system is performing we need to make some comparisons. In particular we can evaluate different models, sampling and so on.
Possiamo provare dei modelli come TableGPT2, TaPaS, poi vari modelli open e close sourced con sampling. Dopodichè possiamo usare un judge LLM. Potremmo anche vedere come performiamo su ToTTo, ma non è il nostro scopo principale.

First testing: we test o1-mini with sampling instead of data profiles